# Homework 9: Tokenization, Quantization, and Mixture of Experts

---

### Objectives

In this assignment you will practice three key concepts from Lecture 9:

1. **Byte Pair Encoding (BPE)** -- Build a subword tokenizer from scratch to understand how modern LLMs construct their vocabularies.
2. **Quantization** -- Implement FP32-to-INT8 weight quantization on a small neural network and measure the memory vs. precision trade-off.
3. **Mixture of Experts (MoE)** -- Build a toy MoE layer with a learned router and top-k sparse activation.

---

### Skills You Will Practice

- Implementing the BPE merge algorithm from first principles
- Computing scale factors and quantizing / dequantizing tensors
- Building gating networks and sparse expert routing with PyTorch
- Analyzing trade-offs: vocabulary size, memory savings, compute efficiency

---

### Guidelines

- **Do not modify** the structure or logic of the provided code -- your task is to understand and complete it.
- All code runs on a laptop CPU. No GPU or large model downloads required.
- Total: **25 points**

---

### Setup

In [5]:
import subprocess, sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

install_if_missing("torch")
install_if_missing("numpy")

import warnings
warnings.filterwarnings("ignore")

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter

torch.manual_seed(42)

## Problem 1: Byte Pair Encoding (BPE) Tokenizer from Scratch (35 points)

### Background

In this problem you will implement the BPE algorithm that models like GPT-4 use to build their vocabularies.
Instead of using a library, you will code each step by hand.

### Recap from lecture

1. Start with all individual characters as your vocabulary.
2. Count every pair of adjacent symbols in your corpus.
3. Merge the most frequent pair into a new symbol.
4. Repeat until you reach your target number of merges.

In [7]:
# Corpus setup (provided -- do not modify)

def build_corpus(text):
    """Convert raw text into the BPE corpus format.
    Each word becomes a tuple of characters + end-of-word marker,
    paired with its frequency."""
    words = text.strip().split()
    freq = Counter(words)
    corpus = {}
    for word, count in freq.items():
        symbols = tuple(list(word) + ["</w>"])
        corpus[symbols] = count
    return corpus

raw_text = "low low low low low lower lower newest newest newest newest newest newest widest widest widest"
corpus = build_corpus(raw_text)

print("Initial corpus:")
for word, count in corpus.items():
    print(f"  {' '.join(word)} : {count}")

Initial corpus:
  l o w </w> : 5
  l o w e r </w> : 2
  n e w e s t </w> : 6
  w i d e s t </w> : 3


In [8]:
# TODO: Implement get_pair_counts (3 points)
#
# Count the frequency of each adjacent pair of symbols across the corpus.
#
# Hints:
# - Loop through each word (tuple of symbols) in the corpus
# - For each word, iterate through indices i from 0 to len(symbols)-2
# - The pair at position i is (symbols[i], symbols[i+1])
# - Add the word's frequency to that pair's count
# - Use collections.Counter to accumulate counts
# - Return the Counter

def get_pair_counts(corpus):
    pairs = Counter()
    for symbols, freq in corpus.items():
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] += freq
            
    return pairs

# Test it
pair_counts = get_pair_counts(corpus)
print("Top 5 pairs:")
for pair, count in pair_counts.most_common(5):
    print(f"  {pair} : {count}")

Top 5 pairs:
  ('e', 's') : 9
  ('s', 't') : 9
  ('t', '</w>') : 9
  ('w', 'e') : 8
  ('l', 'o') : 7


In [9]:
# TODO: Implement merge_pair (3 points)
#
# Given a corpus and a pair to merge, produce a new corpus where
# every occurrence of that pair is replaced by the merged symbol.
#
# Hints:
# - Create the merged symbol: merged = pair[0] + pair[1]
# - For each word in corpus, scan through its symbols with a while loop
# - If symbols[i] and symbols[i+1] match the pair, append merged and skip i+1
# - Otherwise append symbols[i] and advance by 1
# - Store the new tuple in new_corpus with the same frequency

def merge_pair(corpus, pair):
    new_corpus = {}
    merged = pair[0] + pair[1]

    for symbols, freq in corpus.items():
        i = 0
        new_symbols = []
        
        while i < len(symbols)-1:
            if (symbols[i] + symbols[i+1]) == merged:
                new_symbols.append(merged)
                i+=2
            else:
                new_symbols.append(symbols[i])
                i+=1
        new_corpus[tuple(new_symbols)] = freq
    return new_corpus

In [10]:
# TODO: Implement run_bpe (2 points)
#
# Run the full BPE algorithm for num_merges iterations.
#
# Hints:
# - Initialize vocab with all unique symbols from the corpus
# - In each iteration:
#   1. Call get_pair_counts(corpus)
#   2. Find the most common pair using .most_common(1)[0]
#   3. Call merge_pair(corpus, best_pair) to update the corpus
#   4. Add the merged symbol to vocab
#   5. Append best_pair to the merges list
# - Print info at each step so you can trace the algorithm

def run_bpe(corpus, num_merges):
    merges = []
    vocab = set()
    for symbols in corpus.keys():
        for s in symbols:
            vocab.add(s)
    print(f"Initial vocab ({len(vocab)}): {sorted(vocab)}")
    print()

    for i in range(num_merges):
        # TODO: Your code here
        # 1. Get pair counts
        pair_counts = get_pair_counts(corpus)
        if not pair_counts:
            break
        # 2. Find the most common pair
        most_cmn_pair = max(pair_counts, key=pair_counts.get)
        # 3. Merge it in the corpus
        corpus = merge_pair(corpus, most_cmn_pair)
        # 4. Update vocab and merges list
        merged_symbol = most_cmn_pair[0] + most_cmn_pair[1]
        vocab.add(merged_symbol)
        merges.append(most_cmn_pair)
        # 5. Print the merge info
        print(f"Updated vocabulary ({len(vocab)}): {sorted(vocab)}")
        pass

    return corpus, merges, vocab

final_corpus, merges, vocab = run_bpe(corpus, num_merges=10)

Initial vocab (11): ['</w>', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']

Updated vocabulary (12): ['</w>', 'd', 'e', 'es', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
Updated vocabulary (13): ['</w>', 'd', 'e', 'es', 'est', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
Updated vocabulary (14): ['</w>', 'd', 'e', 'es', 'est', 'i', 'l', 'lo', 'n', 'o', 'r', 's', 't', 'w']
Updated vocabulary (15): ['</w>', 'd', 'e', 'es', 'est', 'i', 'l', 'lo', 'n', 'ne', 'o', 'r', 's', 't', 'w']
Updated vocabulary (16): ['</w>', 'd', 'e', 'es', 'est', 'i', 'l', 'lo', 'n', 'ne', 'o', 'r', 's', 't', 'w', 'wi']


In [11]:
# TODO: Implement encode_word (2 points)
#
# Apply the learned BPE merges (in order) to tokenize a new word.
#
# Hints:
# - Start with symbols = list(word) + ["</w>"]
# - For each merge pair (in the order they were learned):
#   - Scan through symbols with a while loop
#   - If symbols[i] and symbols[i+1] match the pair, combine them
#   - Otherwise advance i by 1
# - Return the final list of symbols

def encode_word(word, merges):
    symbols = list(word) + ["</w>"]
    # TODO: Your code here
    for merge in merges:
        new_symbols = []
        i = 0
        while i < len(symbols):
            if i < len(symbols)-1 and (symbols[i] + symbols[i+1]) == merge:
                pair = symbols[i] + symbols[i+1]
                new_symbols.append(pair)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols

test_words = ["low", "lower", "newest", "widest", "new", "lowest"]
print("Tokenization results:")
for w in test_words:
    tokens = encode_word(w, merges)
    print(f'  "{w}" -> {tokens}')

print(f"\nFinal vocabulary ({len(vocab)} tokens): {sorted(vocab)}")

Tokenization results:
  "low" -> ['l', 'o', 'w', '</w>']
  "lower" -> ['l', 'o', 'w', 'e', 'r', '</w>']
  "newest" -> ['n', 'e', 'w', 'e', 's', 't', '</w>']
  "widest" -> ['w', 'i', 'd', 'e', 's', 't', '</w>']
  "new" -> ['n', 'e', 'w', '</w>']
  "lowest" -> ['l', 'o', 'w', 'e', 's', 't', '</w>']

Final vocabulary (16 tokens): ['</w>', 'd', 'e', 'es', 'est', 'i', 'l', 'lo', 'n', 'ne', 'o', 'r', 's', 't', 'w', 'wi']


## Problem 2: Understanding Quantization (35 points)

### Background

In this problem you will implement **FP32 to INT8 quantization** from scratch on a small neural network.
No large models are needed -- we use a simple 2-layer MLP with about 10K parameters.

### Recap from lecture

- FP32 weights use 4 bytes each. INT8 uses 1 byte -- a **75% memory reduction**.
- Scale factor: ``scale = max_abs_weight / 127``
- Quantize: ``int8_value = round(weight / scale)``, clamped to [-128, 127]
- Dequantize: ``approx_weight = int8_value * scale``

In [12]:
# Small MLP for testing (provided -- do not modify)

class SmallMLP(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = SmallMLP()
total_params = sum(p.numel() for p in model.parameters())
total_bytes = total_params * 4
print(f"Model parameters: {total_params:,}")
print(f"Model size (FP32): {total_bytes:,} bytes")

Model parameters: 9,610
Model size (FP32): 38,440 bytes


In [13]:
# TODO Part A: Implement quantize_tensor_int8 (3 points)
#
# Hints:
# - Find the maximum absolute value: tensor.abs().max().item()
# - Compute scale = max_abs / 127.0
# - Quantize: torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
# - Return (quantized_tensor, scale)

def quantize_tensor_int8(tensor):
    # TODO: Your code here
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0
    quantized =  torch.clamp(torch.round(tensor / scale), -128, 127).to(torch.int8)
    return quantized, scale


# TODO Part B: Implement dequantize_tensor (2 points)
# Hint: Cast quantized to float, then multiply by scale

def dequantize_tensor(quantized, scale):
    # TODO: Your code here
    return quantized.float() * scale


# Test on a sample tensor
sample = torch.tensor([0.72, -1.31, 0.05, -0.68, 1.02])
q, s = quantize_tensor_int8(sample)
dq = dequantize_tensor(q, s)

print(f"Original:     {sample.tolist()}")
print(f"Scale factor: {s:.6f}")
print(f"Quantized:    {q.tolist()}")
print(f"Dequantized:  {dq.tolist()}")
print(f"Error (MSE):  {F.mse_loss(sample, dq).item():.8f}")

Original:     [0.7200000286102295, -1.309999942779541, 0.05000000074505806, -0.6800000071525574, 1.0199999809265137]
Scale factor: 0.010315
Quantized:    [70, -127, 5, -66, 99]
Dequantized:  [0.7220472097396851, -1.309999942779541, 0.05157480016350746, -0.6807873845100403, 1.0211809873580933]
Error (MSE):  0.00000174


In [14]:
# TODO Part C: Implement quantize_model (2 points)
#
# Hints:
# - Loop through model.named_parameters()
# - Call quantize_tensor_int8(param.data) on each
# - Store in a dict: name -> (quantized_tensor, scale)

def quantize_model(model):
    quantized_params = {}
    # TODO: Your code here
    for name, param in model.named_parameters():
        quantized, scale = quantize_tensor_int8(param.data)
        quantized_params[name] = (quantized, scale)

    return quantized_params

quantized_params = quantize_model(model)
print("Quantized parameters:")
for name, (q, s) in quantized_params.items():
    print(f"  {name}: shape={list(q.shape)}, scale={s:.6f}, dtype={q.dtype}")

Quantized parameters:
  fc1.weight: shape=[128, 64], scale=0.000984, dtype=torch.int8
  fc1.bias: shape=[128], scale=0.000978, dtype=torch.int8
  fc2.weight: shape=[10, 128], scale=0.000696, dtype=torch.int8
  fc2.bias: shape=[10], scale=0.000616, dtype=torch.int8


In [15]:
# TODO Part D: Compare original vs quantized model (3 points)
#
# Hints:
# - FP32 bytes = sum of numel * 4 for each parameter
# - INT8 bytes = sum of numel * 1 for each quantized param + 4 bytes per scale factor
# - Per-layer MSE: dequantize each param, compare with F.mse_loss
# - For inference: create a new SmallMLP, copy dequantized weights, run same input

fp32_bytes = sum(p.numel() * 4 for p in model.parameters())
# TODO: compute int8_bytes
int8_bytes = sum(q.numel() + 4 for name, (q, s) in quantized_params.items())

print(f"FP32 model size: {fp32_bytes:,} bytes")
print(f"INT8 model size: {int8_bytes:,} bytes")
# TODO: print compression ratio
print(f"Compression ratio: {fp32_bytes / int8_bytes:.2f}x")

# TODO: Compute per-layer MSE
print("\nPer-layer reconstruction error (MSE):")
for name, param in model.named_parameters():
    q, s = quantized_params[name]
    # TODO: dequantize and compute MSE
    mse = F.mse_loss(dequantize_tensor(q, s), param.data)
    print(f"  {name}: MSE = {mse:.8f}")

# TODO: Compare inference outputs
test_input = torch.randn(1, 64)
with torch.no_grad():
    original_output = model(test_input)

# TODO: Build a dequantized model and compare outputs
dq_model = SmallMLP()
with torch.no_grad():
    for name, param in dq_model.named_parameters():
        q, s = quantized_params[name]
        param.copy_(dequantize_tensor(q, s)) # replaces the random weights in dq_model with the dequantized weights
with torch.no_grad():
    dq_output = dq_model(test_input)
print(f"Output MSE: {F.mse_loss(original_output, dq_output).item():.8f}")

FP32 model size: 38,440 bytes
INT8 model size: 9,626 bytes
Compression ratio: 3.99x

Per-layer reconstruction error (MSE):
  fc1.weight: MSE = 0.00000008
  fc1.bias: MSE = 0.00000007
  fc2.weight: MSE = 0.00000004
  fc2.bias: MSE = 0.00000003
Output MSE: 0.00000065


## Problem 3: Mixture of Experts (MoE) Layer (20 points)

### Background

In this problem you will build a **toy Mixture of Experts layer** to understand how sparse
expert routing works in models like Mixtral 8x7B.

### Recap from lecture

- A standard Transformer block has one shared FFN. In MoE, the FFN is replaced by multiple expert FFNs.
- A **Router** (small linear layer + softmax) decides which experts to activate for each input.
- **Top-K routing**: only K of N experts are activated per token, making inference efficient.
- Total parameters = all experts loaded. Active parameters = shared layers + K selected experts.

In [16]:
# Expert: a simple 2-layer FFN (provided -- do not modify)

class Expert(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x):
        return self.net(x)

In [17]:
# TODO: Implement Router and MoELayer (5 points)
#
# Router hints:
# - __init__: create a single nn.Linear(input_dim, num_experts)
# - forward: return F.softmax(self.gate(x), dim=-1)
#
# MoELayer forward hints:
# - Get gate_scores from self.router(x)        -> shape (batch, num_experts)
# - Use torch.topk(gate_scores, self.top_k)     -> topk_scores, topk_indices
# - Normalize: topk_scores / topk_scores.sum(dim=-1, keepdim=True)
# - Initialize output = torch.zeros_like(x)
# - For each of the top_k selected experts:
#     - Get expert index for each item in the batch
#     - Run that expert on the input
#     - Multiply by the gating weight and add to output
# - Return (output, gate_scores, topk_indices)

class Router(nn.Module):
    def __init__(self, input_dim, num_experts):
        super().__init__()
        # TODO: Define the gating layer
        self.gate = nn.Linear(input_dim, num_experts)


    def forward(self, x):
        # TODO: Return softmax routing weights
        return F.softmax(self.gate(x), dim=-1)


class MoELayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_experts=4, top_k=2):
        super().__init__()
        self.experts = nn.ModuleList([Expert(input_dim, hidden_dim) for _ in range(num_experts)])
        self.router = Router(input_dim, num_experts)
        self.top_k = top_k
        self.num_experts = num_experts

    def forward(self, x):
        # TODO: Implement top-k expert routing
        # 1. Get gate scores from router
        gate_scores = self.router(x)
        # 2. Select top-k experts with torch.topk
        top_scores, top_indices = torch.topk(gate_scores, self.top_k)
        # 3. Normalize top-k scores
        normalized_scores = top_scores / sum(top_scores)
        
        # 4. Run selected experts and combine outputs
        output = torch.zeros_like(x)
        for k in range(self.top_k): # for each of the top experts batch (there is a batch of experts, one for each datapoint)
            top_experts_indices = top_indices[:, k] # get the expert for that dataset (column)
            weights = top_scores[:, k].unsqueeze(-1)

            for expert_id, expert in enumerate(self.experts):
                mask =  top_experts_indices == expert_id 
                # mask routes to the correct expert
                if mask.any():
                    expert_output = expert(x[mask])
                    output[mask] += weights[mask] * expert_output
        return output, gate_scores, top_indices


input_dim = 32
hidden_dim = 64
moe = MoELayer(input_dim, hidden_dim, num_experts=4, top_k=2)

In [18]:
# Demonstrate routing decisions (run after implementing MoELayer)

torch.manual_seed(0)
batch = torch.randn(5, input_dim)

output, gate_scores, topk_indices = moe(batch)

print("Routing decisions for 5 inputs:")
print("=" * 50)
for i in range(5):
    scores = gate_scores[i].detach().tolist()
    selected = topk_indices[i].detach().tolist()
    score_str = ", ".join(f"{s:.3f}" for s in scores)
    print(f"Input {i}: gate scores = [{score_str}]")
    print(f"          selected experts = {selected}")
    print()

# Parameter analysis
total_params = sum(p.numel() for p in moe.parameters())
expert_params = sum(p.numel() for e in moe.experts for p in e.parameters())
single_expert = sum(p.numel() for p in moe.experts[0].parameters())
router_params = sum(p.numel() for p in moe.router.parameters())
active_params = router_params + 2 * single_expert  # top-2 routing

print("Parameter Analysis:")
print(f"  Total parameters (loaded):  {total_params:,}")
print(f"  Router parameters:          {router_params:,}")
print(f"  Per-expert parameters:      {single_expert:,}")
print(f"  Active parameters (top-2):  {active_params:,}")
pct = active_params / total_params * 100
print(f"  Sparsity ratio:             {pct:.1f}% of params active per input")

Routing decisions for 5 inputs:
Input 0: gate scores = [0.191, 0.299, 0.255, 0.255]
          selected experts = [1, 2]

Input 1: gate scores = [0.357, 0.263, 0.214, 0.166]
          selected experts = [0, 1]

Input 2: gate scores = [0.195, 0.277, 0.324, 0.204]
          selected experts = [2, 1]

Input 3: gate scores = [0.171, 0.373, 0.305, 0.151]
          selected experts = [1, 2]

Input 4: gate scores = [0.341, 0.108, 0.283, 0.269]
          selected experts = [0, 2]

Parameter Analysis:
  Total parameters (loaded):  16,900
  Router parameters:          132
  Per-expert parameters:      4,192
  Active parameters (top-2):  8,516
  Sparsity ratio:             50.4% of params active per input


## Problem 4: Microsoft Foundry LLM Endpoint Setup (10 points)

### Background

In this problem you will connect to the OpenAI Model endpont hosted on Microsoft Foundry

### Recap from lecture
- Following the instructions provided at the end of the lecture, locate the LLM endpoint specific to your account on the Foundry page. Replace "YOUR ENDPOINT" in the code with your unique URL.

- Upon running the cell, you will be prompted to log in to Microsoft Foundry. Once authenticated, a successful connection will return the following confirmation:

"[SUCCESS] Response from model: Hello! Yes, your connection via Interactive Browser Authentication is confirmed."

- If authentication fails, please restart your kernel and try again.. 

In [19]:
! pip install openai azure-identity


import os
from azure.identity import DeviceCodeCredential, get_bearer_token_provider
from openai import AzureOpenAI
# 1. Auth (This will use the token you just generated)
credential = DeviceCodeCredential()
token_provider = get_bearer_token_provider(
    credential, 
    "https://cognitiveservices.azure.com/.default"
)

# 2. Connect using the correct OpenAI-specific base URL
client = AzureOpenAI(
    azure_endpoint="https://mlearn530nusmith-aiservices.services.ai.azure.com/",
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01"
)

# 3. Fire the test request
try:
    print("Sending test prompt to GPT-4o...")
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "user", "content": "Hello! Confirming the connection works via Interactive Browser Auth."}
        ]
    )
    print("\n[SUCCESS] Response from model:")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"\n[FAILURE] Connection failed: {e}")

Sending test prompt to GPT-4o...
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code ANW24MJ2P to authenticate.

[SUCCESS] Response from model:
Hello! The connection via Interactive Browser Authentication appears to have been established successfully. I'm here to assist you—just let me know what you need! 😊
